# 🎬 Explorateur de chaîne YouTube — @NuitsdeSavoir

Ce notebook récupère toutes les vidéos de la chaîne **Nuits de Savoir** et génère un résumé à coller dans Claude pour obtenir des recommandations personnalisées.

**Étapes :**
1. Entre ta clé API YouTube ci-dessous
2. Exécute toutes les cellules (Ctrl+F9 ou Menu → Exécution → Tout exécuter)
3. Copie le résumé généré à la fin et colle-le dans ta conversation Claude

In [ ]:
# ============================================================
# ÉTAPE 1 : Entre ta clé API YouTube ici
# ============================================================
API_KEY = "COLLE_TA_CLE_API_ICI"  # Remplace par ta clé

CHANNEL_HANDLE = "@NuitsdeSavoir"
CHANNEL_URL = "https://youtube.com/@nuitsdesavoir"

In [ ]:
# ============================================================
# ÉTAPE 2 : Installation des dépendances
# ============================================================
import subprocess
subprocess.run(["pip", "install", "requests", "pandas"], capture_output=True)

import requests
import pandas as pd
from datetime import datetime

print("✅ Dépendances chargées")

In [ ]:
# ============================================================
# ÉTAPE 3 : Trouver l'ID de la chaîne
# ============================================================
def get_channel_id(api_key, handle):
    """Récupère l'ID de chaîne depuis le handle @NuitsdeSavoir"""
    # Méthode 1 : recherche par handle (API v3)
    url = "https://www.googleapis.com/youtube/v3/channels"
    params = {
        "part": "id,snippet,contentDetails",
        "forHandle": handle.lstrip("@"),
        "key": api_key
    }
    r = requests.get(url, params=params)
    data = r.json()

    if data.get("items"):
        channel = data["items"][0]
        channel_id = channel["id"]
        uploads_playlist = channel["contentDetails"]["relatedPlaylists"]["uploads"]
        name = channel["snippet"]["title"]
        description = channel["snippet"].get("description", "")[:200]
        return channel_id, uploads_playlist, name, description
    else:
        raise Exception(f"Chaîne introuvable. Réponse : {data}")

channel_id, uploads_playlist_id, channel_name, channel_desc = get_channel_id(API_KEY, CHANNEL_HANDLE)
print(f"✅ Chaîne trouvée : {channel_name}")
print(f"   ID : {channel_id}")
print(f"   Playlist uploads : {uploads_playlist_id}")
print(f"   Description : {channel_desc[:100]}...")

In [ ]:
# ============================================================
# ÉTAPE 4 : Récupérer toutes les vidéos (via playlist uploads)
# Note : on utilise playlistItems (1 unité/appel) plutôt que
# search (100 unités/appel) pour économiser le quota
# ============================================================
def get_all_video_ids(api_key, playlist_id):
    """Récupère tous les IDs de vidéos d'une playlist avec pagination"""
    video_ids = []
    url = "https://www.googleapis.com/youtube/v3/playlistItems"
    next_page_token = None

    while True:
        params = {
            "part": "contentDetails",
            "playlistId": playlist_id,
            "maxResults": 50,
            "key": api_key
        }
        if next_page_token:
            params["pageToken"] = next_page_token

        r = requests.get(url, params=params)
        data = r.json()

        if "error" in data:
            raise Exception(f"Erreur API : {data['error']['message']}")

        for item in data.get("items", []):
            video_ids.append(item["contentDetails"]["videoId"])

        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break

        print(f"   {len(video_ids)} vidéos récupérées...", end="\r")

    return video_ids

print("Récupération des IDs de vidéos...")
video_ids = get_all_video_ids(API_KEY, uploads_playlist_id)
print(f"\n✅ {len(video_ids)} vidéos trouvées au total")

In [ ]:
# ============================================================
# ÉTAPE 5 : Récupérer les détails de chaque vidéo
# (titre, description, tags, durée, vues, date)
# ============================================================
def get_videos_details(api_key, video_ids):
    """Récupère les détails pour une liste d'IDs (par batch de 50)"""
    videos = []
    url = "https://www.googleapis.com/youtube/v3/videos"

    # On traite par batch de 50 (limite de l'API)
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        params = {
            "part": "snippet,statistics,contentDetails",
            "id": ",".join(batch),
            "key": api_key
        }
        r = requests.get(url, params=params)
        data = r.json()

        if "error" in data:
            raise Exception(f"Erreur API : {data['error']['message']}")

        for item in data.get("items", []):
            snippet = item["snippet"]
            stats = item.get("statistics", {})
            content = item.get("contentDetails", {})

            # Convertir la durée ISO 8601 en minutes
            duration_iso = content.get("duration", "PT0S")
            import re
            h = int(re.search(r'(\d+)H', duration_iso).group(1)) if re.search(r'(\d+)H', duration_iso) else 0
            m = int(re.search(r'(\d+)M', duration_iso).group(1)) if re.search(r'(\d+)M', duration_iso) else 0
            s = int(re.search(r'(\d+)S', duration_iso).group(1)) if re.search(r'(\d+)S', duration_iso) else 0
            duration_min = round(h * 60 + m + s / 60, 1)

            videos.append({
                "id": item["id"],
                "titre": snippet.get("title", ""),
                "date": snippet.get("publishedAt", "")[:10],
                "description": snippet.get("description", "")[:300],
                "tags": ", ".join(snippet.get("tags", [])),
                "vues": int(stats.get("viewCount", 0)),
                "likes": int(stats.get("likeCount", 0)),
                "duree_min": duration_min,
                "url": f"https://youtu.be/{item['id']}"
            })

        print(f"   Détails : {min(i+50, len(video_ids))}/{len(video_ids)} vidéos", end="\r")

    return videos

print("Récupération des détails des vidéos...")
videos = get_videos_details(API_KEY, video_ids)
df = pd.DataFrame(videos).sort_values("date", ascending=False).reset_index(drop=True)
print(f"\n✅ Détails récupérés pour {len(df)} vidéos")

In [ ]:
# ============================================================
# ÉTAPE 6 : Aperçu du catalogue
# ============================================================
print(f"📺 Chaîne : {channel_name}")
print(f"📊 Nombre de vidéos : {len(df)}")
print(f"📅 De {df['date'].min()} à {df['date'].max()}")
print(f"👁  Vues totales : {df['vues'].sum():,}")
print(f"⏱  Durée moyenne : {df['duree_min'].mean():.0f} min")
print()
print("Top 5 vidéos les plus vues :")
top5 = df.nlargest(5, 'vues')[['titre', 'vues', 'duree_min', 'date']]
for _, row in top5.iterrows():
    print(f"  - {row['titre'][:60]} ({row['vues']:,} vues, {row['duree_min']}min)")

In [ ]:
# ============================================================
# ÉTAPE 7 : Générer le résumé à coller dans Claude
# ============================================================
lines = []
lines.append(f"Voici toutes les vidéos de la chaîne YouTube '{channel_name}' ({len(df)} vidéos).")
lines.append(f"Description de la chaîne : {channel_desc}")
lines.append("")
lines.append("LISTE DES VIDÉOS (titre | date | durée | vues | tags | début de description)")
lines.append("=" * 80)

for _, row in df.iterrows():
    line = f"- [{row['date']}] {row['titre']} | {row['duree_min']}min | {row['vues']:,} vues"
    if row['tags']:
        line += f" | Tags: {row['tags'][:80]}"
    if row['description']:
        desc_short = row['description'].replace('\n', ' ')[:150]
        line += f" | {desc_short}"
    line += f" | {row['url']}"
    lines.append(line)

summary = "\n".join(lines)

print(summary[:3000])  # Aperçu des 3000 premiers caractères
print("...")
print(f"\n✅ Résumé généré ({len(summary)} caractères)")

In [ ]:
# ============================================================
# ÉTAPE 8 : Sauvegarder le résumé dans un fichier texte
# (plus facile à copier depuis le téléphone)
# ============================================================
filename = "nuitsdesavoir_pour_claude.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(summary)

print(f"✅ Fichier sauvegardé : {filename}")
print()
print("📋 INSTRUCTIONS :")
print("1. Télécharge le fichier depuis le panneau 'Fichiers' (icône dossier à gauche)")
print("2. Ouvre-le et copie son contenu")
print("3. Colle-le dans ta conversation Claude")
print("4. Dis à Claude : 'Voici les vidéos de la chaîne. Aide-moi à trouver celles qui me correspondent.'")

# Téléchargement direct depuis Colab
try:
    from google.colab import files
    files.download(filename)
    print("\n⬇️  Téléchargement lancé automatiquement")
except ImportError:
    print("(Téléchargement manuel depuis le panneau Fichiers)")